Import Libraries

In [1]:
import pandas as pd
from pathlib import Path
import statsmodels.api as sm

Set-up

In [2]:
# Get root directory and data directory
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_PATH = (PROJECT_ROOT / "data" / "processed" / "retailhero_causal_forest_features.csv")

RANDOM_STATE = 1213

In [3]:
# Load data
print("=" * 60)
print("Logistic Regression - Simple Interaction Models")
print("=" * 60)

print(f"\nLoading data from:")
print(PROCESSED_DATA_PATH.resolve())

df = pd.read_csv(PROCESSED_DATA_PATH)

print(f"\nFull dataset shape: {df.shape}")

Logistic Regression - Simple Interaction Models

Loading data from:
C:\Users\szepi\OneDrive\Documents\dse4101\DSE4101-CausalForest-Grp2\data\Processed\retailhero_causal_forest_features.csv

Full dataset shape: (200039, 25)


In [4]:
# Define Y, D, X
D = df["treatment"] 
Y = df["target"]

X = df.drop(columns=["client_id", "treatment", "target","gender_U"]) 

# Pre-specified Logistic Regression

## Model 1: Prior Total Spending
Y ~ D + total_spend + D*total_spend

In [5]:
model1_df = pd.DataFrame({
    "target": Y,
    "treatment": D,
    "total_spend": X["total_spend"]
})

model1_df["treatment_x_total_spend"] = (model1_df["treatment"] * model1_df["total_spend"])

X_logit1 = model1_df[["treatment", "total_spend", "treatment_x_total_spend"]]
X_logit1 = sm.add_constant(X_logit1)

logit_model1 = sm.Logit(model1_df["target"],X_logit1).fit()

print("\n" + "=" * 60)
print("Pre-specified Logistic Regression: Prior Total Spending")
print("=" * 60)

print(logit_model1.summary())

Optimization terminated successfully.
         Current function value: 0.619779
         Iterations 6

Pre-specified Logistic Regression: Prior Total Spending
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               200039
Model:                          Logit   Df Residuals:                   200035
Method:                           MLE   Df Model:                            3
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                 0.06676
Time:                        18:32:18   Log-Likelihood:            -1.2398e+05
converged:                       True   LL-Null:                   -1.3285e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                    

very small p-value

There is statistically significant evidence that treatment-effect heterogeneity is associated with prior total spending.

## Model 2: Number of Transactions
Y ~ D + num_transactions + D*num_transactions

In [ ]:
model2_df = pd.DataFrame({
    "target": Y,
    "treatment": D,
    "num_transactions": X["num_transactions"]
})

model2_df["treatment_x_num_transactions"] = (model2_df["treatment"] * model2_df["num_transactions"])

X_logit2 = model2_df[["treatment", "num_transactions", "treatment_x_num_transactions"]]
X_logit2 = sm.add_constant(X_logit2)

logit_model2 = sm.Logit(model2_df["target"],X_logit2).fit()

print("\n" + "=" * 70)
print("Pre-specified Logistic Regression: Prior Number of Transactions")
print("=" * 70)

print(logit_model2.summary())

Optimization terminated successfully.
         Current function value: 0.572348
         Iterations 6

Pre-specified Logistic Regression: Pre-Treatment Number of Transactions
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               200039
Model:                          Logit   Df Residuals:                   200035
Method:                           MLE   Df Model:                            3
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                  0.1382
Time:                        18:32:21   Log-Likelihood:            -1.1449e+05
converged:                       True   LL-Null:                   -1.3285e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                   coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------

p-value = 0.081 > 0.05

There is no statistically significant evidence that treatment effect heterogenity is associated with pre-treatment number of transactions.

## Model 3: Purchase Recency
Y ~ D + days_since_last_purchase + D*days_since_last_purchase

In [7]:
model3_df = pd.DataFrame({
    "target": Y,
    "treatment": D,
    "days_since_last_purchase": X["days_since_last_purchase"]
})

model3_df["treatment_x_days_since_last_purchase"] = (model3_df["treatment"] * model3_df["days_since_last_purchase"])

X_logit3 = model3_df[["treatment", "days_since_last_purchase", "treatment_x_days_since_last_purchase"]]
X_logit3 = sm.add_constant(X_logit3)

logit_model3 = sm.Logit(model3_df["target"],X_logit3).fit()

print("\n" + "=" * 70)
print("Pre-specified Logistic Regression: Pre-Treatment Purchase Recency")
print("=" * 70)

print(logit_model3.summary())

Optimization terminated successfully.
         Current function value: 0.611638
         Iterations 5

Pre-specified Logistic Regression: Pre-Treatment Purchase Recency
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               200039
Model:                          Logit   Df Residuals:                   200035
Method:                           MLE   Df Model:                            3
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                 0.07902
Time:                        18:32:24   Log-Likelihood:            -1.2235e+05
converged:                       True   LL-Null:                   -1.3285e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------

p-value = 0.033 < 0.05 

There is statistically significant evidence that treatment effect heterogenity is associated with pre-treatment purchase recency.

# LASSO-informed Logistic Regression

$$
\operatorname{logit}\left[P(Y_i=1 \mid D_i,X_i)\right]
=
\beta_0
+\beta_D D_i
+\boldsymbol{\beta}'X_i
+\boldsymbol{\gamma}'(D_iX_i)
$$

where:

- $Y_i$ is the binary purchase outcome;
- $D_i$ is the treatment indicator;
- $X_i$ contains the customer characteristics selected by LASSO;
- $D_iX_i$ contains the corresponding treatment-covariate interactions; and
- $\boldsymbol{\gamma}$ captures systematic heterogeneity in treatment response across customer characteristics.

Load Lasso-selected features

In [9]:
LASSO_FEATURES_PATH = (PROJECT_ROOT / "data" / "output" / "lasso_selected_features.csv")
lasso_features = pd.read_csv(LASSO_FEATURES_PATH)["feature"].tolist()

print("LASSO-selected features:")
for feature in lasso_features:
    print(f"- {feature}")

LASSO-selected features:
- express_points_spent
- age
- gender_F
- num_stores_visited
- spend_std


In [10]:
lasso_logit_df = df.copy()
interaction_cols = []

# create interaction terms for LASSO-selected features
for feature in lasso_features:
    interaction_col_name = f"treatment_x_{feature}"
    lasso_logit_df[interaction_col_name] = lasso_logit_df["treatment"] * lasso_logit_df[feature]
    interaction_cols.append(interaction_col_name)

# model predictiors: treatment + selected main effected + selected treatment interactions

model_cols = (["treatment"] + lasso_features + interaction_cols)

x_lasso_logit = lasso_logit_df[model_cols]
y_lasso_logit = lasso_logit_df["target"]
x_lasso_logit = sm.add_constant(x_lasso_logit) #add intercept

print(f"number of predictors: {len(model_cols)}")
print("model predictors:")
for col in model_cols:
    print(f" - {col}")

lasso_informed_logit = sm.Logit(y_lasso_logit, x_lasso_logit).fit()
print(lasso_informed_logit.summary())


number of predictors: 11
model predictors:
 - treatment
 - express_points_spent
 - age
 - gender_F
 - num_stores_visited
 - spend_std
 - treatment_x_express_points_spent
 - treatment_x_age
 - treatment_x_gender_F
 - treatment_x_num_stores_visited
 - treatment_x_spend_std
Optimization terminated successfully.
         Current function value: 0.650831
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                 target   No. Observations:               200039
Model:                          Logit   Df Residuals:                   200027
Method:                           MLE   Df Model:                           11
Date:                Sat, 12 Sep 2026   Pseudo R-squ.:                 0.02001
Time:                        18:32:31   Log-Likelihood:            -1.3019e+05
converged:                       True   LL-Null:                   -1.3285e+05
Covariance Type:            nonrobust   LLR p-value:                     

At the 5% significance level, four of the five treatment interaction terms are statistically significant, suggesting treatment-effect heterogeneity across several customer characteristics. The interaction with number of stores visited is not statistically significant (\(p=0.092\)).